This colab notebook demonstrates how to convert, quantize, and deploy the Imp Vision Language Model on Android using the MLC framework for on-device inference.

#### Step 1: Clone MLC-Imp and Vision LM repositories

In [ ]:
!git clone --recursive https://github.com/MILVLG/mlc-imp.git
!git clone https://github.com/NSTiwari/Mobile-VisionLM.git

Cloning into 'mlc-imp'...
remote: Enumerating objects: 11668, done.
remote: Total 11668 (delta 0), reused 0 (delta 0), pack-reused 11668 (from 1)
Receiving objects: 100% (11668/11668), 35.07 MiB | 26.88 MiB/s, done.
Resolving deltas: 100% (7494/7494), done.
Submodule '3rdparty/argparse' (https://github.com/p-ranav/argparse) registered for path '3rdparty/argparse'
Submodule '3rdparty/googletest' (https://github.com/google/googletest.git) registered for path '3rdparty/googletest'
Submodule '3rdparty/tokenizers-cpp' (https://github.com/mlc-ai/tokenizers-cpp) registered for path '3rdparty/tokenizers-cpp'
Submodule '3rdparty/tvm' (https://github.com/mlc-ai/relax.git) registered for path '3rdparty/tvm'
Cloning into '/content/mlc-imp/3rdparty/argparse'...
remote: Enumerating objects: 3117, done.        
remote: Counting objects: 100% (1007/1007), done.        
remote: Compressing objects: 100% (157/157), done.        
remote: Total 3117 (delta 914), reused 850 (delta 850), pack-reused 2110 (f

#### Step 2: Install additional dependencies

* Rust/Cargo
* LLVM
* Timm

In [ ]:
!sudo apt install llvm
!pip install timm
!sudo curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  binfmt-support libpfm4 libz3-4 libz3-dev llvm-14 llvm-14-dev llvm-14-runtime
  llvm-14-tools llvm-runtime python3-pygments python3-yaml
Suggested packages:
  llvm-14-doc python-pygments-doc ttf-bitstream-vera
The following NEW packages will be installed:
  binfmt-support libpfm4 libz3-4 libz3-dev llvm llvm-14 llvm-14-dev
  llvm-14-runtime llvm-14-tools llvm-runtime python3-pygments python3-yaml
0 upgraded, 12 newly installed, 0 to remove and 29 not upgraded.
Need to get 58.5 MB of archives.
After this operation, 354 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-yaml amd64 5.4.1-1ubuntu1 [129 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 binfmt-support amd64 2.2.1-2 [55.8 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 llvm-14-runtime amd64 

In [ ]:
# Set Cargo to path.
import os
os.environ["PATH"] += ":/root/.cargo/bin"

# Validate Cargo installation.
!cargo --version

cargo 1.85.1 (d73d2caf9 2024-12-31)


#### Step 3: Install TVM from source

In [ ]:
!mkdir -p /content/mlc-imp/3rdparty/tvm/build
!cp /content/mlc-imp/3rdparty/tvm/cmake/config.cmake /content/mlc-imp/3rdparty/tvm/build/
%cd /content/mlc-imp/3rdparty/tvm/build

/content/mlc-imp/3rdparty/tvm/build


In [ ]:
!echo "set(CMAKE_BUILD_TYPE RelWithDebInfo)" >> config.cmake
!echo "set(USE_LLVM \"llvm-config --ignore-libllvm --link-static\")" >> config.cmake
!echo "set(HIDE_PRIVATE_SYMBOLS ON)" >> config.cmake
!echo "set(CMAKE_BUILD_TYPE RelWithDebInfo)" >> config.cmake
!echo "set(USE_CUDA   ON)" >> config.cmake

In [ ]:
!cmake ..

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Hide private symbols...
-- Forbidding undefined symbols in shared library, using -Wl,--no-undefined on platform Linux
-- Build with RPC support...
-- Build with Graph Executor support...
-- Build with profiler...
-- Build with AOT Executor support...
-- Could NOT find GTest (missing: GTEST_LIBRARY GTEST_INCLUDE_DIR GTEST_MAIN_LIBRARY) 
-- Build Alloc alignment set to 64
-- Didn't find the path to CCACHE, disabling ccache
-- Performing Test SUPPORT_CXX17
-- Performing Test SUPPORT_CXX17 -

The cell below builds TVM from scratch, and will take approximately 2 hours to execute.

In [ ]:
!cmake --build . --parallel 4

[  0%] Building CXX object CMakeFiles/tvm_libinfo_objs.dir/src/support/libinfo.cc.o
[  0%] Creating directories for 'project_libbacktrace'
[  0%] No download step for 'project_libbacktrace'
[  0%] No checkout step for 'project_libbacktrace'
[  1%] No update step for 'project_libbacktrace'
[  1%] No patch step for 'project_libbacktrace'
[  1%] Building CXX object CMakeFiles/tvm_objs.dir/src/arith/analyzer.cc.o
[  1%] Building CXX object CMakeFiles/tvm_objs.dir/src/arith/bound_deducer.cc.o
[  1%] Performing configure step for 'project_libbacktrace'
checking build system type... x86_64-pc-linux-gnu
checking host system type... x86_64-pc-linux-gnu
checking target system type... x86_64-pc-linux-gnu
checking for gcc... /usr/bin/cc
checking whether the C compiler works... yes
checking for C compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether we are using the GN

In [ ]:
%cd /content/mlc-imp/3rdparty/tvm/python
!pip install -e .

/content/mlc-imp/3rdparty/tvm/python
Obtaining file:///content/mlc-imp/3rdparty/tvm/python
  Preparing metadata (setup.py) ... done
  Running setup.py develop for tvm


#### Step 4: Validate TVM installation

In [ ]:
!python -c "import tvm; print(tvm.__file__)"

/content/mlc-imp/3rdparty/tvm/python/tvm/__init__.py


#### Step 5: Install MLC-LLM from source

When prompted to provide **TVM_HOME**, enter `/content/mlc-imp/3rdparty/tvm`

Configure the settings as follows:
*   CUDA (y)
*   CUTLASS (n)
*   CUBLAS (n)
*   ROCm (n)
*   Vulkan (n)
*   Metal (Apple M1/M2 GPU) (n)
*   OpenCL (y)
*   FlashInfer (n)

In [ ]:
%cd /content/mlc-imp
!mkdir build

!python3 cmake/gen_cmake_config.py
!cd ..

/content/mlc-imp
Enter TVM_HOME in absolute path. If not specified, 3rdparty/tvm will be used by default: /content/mlc-imp/3rdparty/tvm
Use CUDA? (y/n): y
Use CUTLASS? (y/n): n
Use CUBLAS? (y/n): n
Use ROCm? (y/n): n
Use Vulkan? (y/n): n
Use Metal (Apple M1/M2 GPU) ? (y/n): n
Use OpenCL? (y/n) y
Use FlashInfer? (need CUDA w/ compute capability 80;86;89;90) (y/n): n

Writing the following configuration to config.cmake...
set(TVM_HOME /content/mlc-imp/3rdparty/tvm)
set(CMAKE_BUILD_TYPE RelWithDebInfo)
set(USE_CUDA ON)
set(USE_CUTLASS OFF)
set(USE_CUBLAS OFF)
set(USE_ROCM OFF)
set(USE_VULKAN OFF)
set(USE_METAL OFF)
set(USE_OPENCL ON)
set(USE_THRUST ON)
set(USE_FLASHINFER OFF)



The cell below will take 15 minutes to execute.

In [ ]:
%cd build
!cmake ..
!cmake --build . --parallel $(nproc)

/content/mlc-imp/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Hide private symbols
-- TVM_HOME: /content/mlc-imp/3rdparty/tvm
-- Hide private symbols...
-- Forbidding undefined symbols in shared library, using -Wl,--no-undefined on platform Linux
-- Didn't find the path to CCACHE, disabling ccache
-- Performing Test SUPPORT_CXX17
-- Performing Test SUPPORT_CXX17 - Success
-- VTA build with VTA_HW_PATH=/content/mlc-imp/3rdparty/tvm/3rdparty/vta-hw
-- Build VTA runtime with target: sim
CMake Warning (dev) at 3rdparty/tvm/cmake/

In [ ]:
%cd /content/mlc-imp/python
!pip install -e .

/content/mlc-imp/python
Obtaining file:///content/mlc-imp/python
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.8 MB/s eta 0:00:00
  Running setup.py develop for mlc_llm


#### Step 6: Validate MLC-LLM installation

In [ ]:
!python -c "import mlc_llm; print(mlc_llm)"

<module 'mlc_llm' from '/content/mlc-imp/python/mlc_llm/__init__.py'>


#### Step 7: Download Imp-v1.5-3B-196 weights

In [ ]:
MODEL_NAME = "imp-v1.5-3B-196"
QUANTIZATION = "q4f16_1"
MODEL_TYPE = "imp"

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

!mkdir -p /content/mlc-imp/dist/models/$MODEL_NAME
%cd /content/mlc-imp/dist/models/$MODEL_NAME

/content/mlc-imp/dist/models/imp-v1.5-3B-196


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="MILVLG/Imp-v1.5-3B-196", local_dir="/content/mlc-imp/dist/models/imp-v1.5-3B-196/")

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/914 [00:00<?, ?B/s]

LICENSE:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

configuration_imp.py:   0%|          | 0.00/7.04k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.18k [00:00<?, ?B/s]

vision_encoder.py:   0%|          | 0.00/24.4k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/999k [00:00<?, ?B/s]

'/content/mlc-imp/dist/models/imp-v1.5-3B-196'

#### Step 8: Convert model weights to MLC-LLM compatible format.

In [ ]:
!mlc_llm convert_weight \
  --model-type $MODEL_TYPE \
  /content/mlc-imp/dist/models/$MODEL_NAME/ \
  --quantization $QUANTIZATION \
  -o /content/mlc-imp/dist/libs/

[2025-03-26 19:29:52] INFO auto_config.py:115: Found model configuration: /content/mlc-imp/dist/models/imp-v1.5-3B-196/config.json
[2025-03-26 19:29:55] INFO auto_device.py:79: Found device: cuda:0
[2025-03-26 19:29:58] INFO auto_device.py:88: Not found device: rocm:0
[2025-03-26 19:30:01] INFO auto_device.py:88: Not found device: metal:0
[2025-03-26 19:30:03] INFO auto_device.py:88: Not found device: vulkan:0
[2025-03-26 19:30:06] INFO auto_device.py:88: Not found device: opencl:0
[2025-03-26 19:30:06] INFO auto_device.py:35: Using device: cuda:0
[2025-03-26 19:30:06] INFO auto_weight.py:70: Finding weights in: /content/mlc-imp/dist/models/imp-v1.5-3B-196
[2025-03-26 19:30:06] INFO auto_weight.py:136: Not found Huggingface PyTorch
[2025-03-26 19:30:06] INFO auto_weight.py:143: Found source weight format: huggingface-safetensor. Source configuration: /content/mlc-imp/dist/models/imp-v1.5-3B-196/model.safetensors.index.json
[2025-03-26 19:30:06] INFO auto_weight.py:106: Using source wei

#### Step 9: Generate config files

In [ ]:
!mlc_llm gen_config \
  /content/mlc-imp/dist/models/$MODEL_NAME \
  --quantization $QUANTIZATION \
  --conv-template $MODEL_TYPE \
  -o /content/mlc-imp/dist/libs

[2025-03-26 19:31:19] INFO auto_config.py:115: Found model configuration: /content/mlc-imp/dist/models/imp-v1.5-3B-196/config.json
[2025-03-26 19:31:19] INFO auto_config.py:153: Found model type: imp. Use `--model-type` to override.
[2025-03-26 19:31:19] WARNING gen_config.py:149: Warning: Conversation template is not registered in ConvTemplateRegistry: imp
[2025-03-26 19:31:19] WARNING config.py:99: Warning: Cannot override max_batch_size, because PhiConfig does not have this field
[2025-03-26 19:31:19] INFO gen_config.py:187: [generation_config.json] Setting eos_token_id: 50295
[2025-03-26 19:31:19] INFO gen_config.py:187: [generation_config.json] Setting pad_token_id: 50256
[2025-03-26 19:31:19] INFO gen_config.py:201: Not found tokenizer config: /content/mlc-imp/dist/models/imp-v1.5-3B-196/tokenizer.model
[2025-03-26 19:31:19] INFO gen_config.py:199: Found tokenizer config: /content/mlc-imp/dist/models/imp-v1.5-3B-196/tokenizer.json. Copying to /content/mlc-imp/dist/libs/tokenizer.

#### Step 10: Compile the model to Android format

In [ ]:
!mlc_llm compile \
  /content/mlc-imp/dist/libs/mlc-chat-config.json \
  --device android \
  -o /content/mlc-imp/dist/libs/$MODEL_NAME-$QUANTIZATION-android.tar

[2025-03-26 19:31:22] INFO auto_config.py:69: Found model configuration: /content/mlc-imp/dist/libs/mlc-chat-config.json
[2025-03-26 19:31:22] INFO auto_config.py:153: Found model type: imp. Use `--model-type` to override.
[2025-03-26 19:31:22] WARNING auto_target.py:361: --system-lib-prefix is automatically picked from the filename, imp_q4f16_1_, this allows us to use the filename as the model_lib in android/iOS builds. Please avoid renaming the .tar file when uploading the prebuilt.
Compiling with arguments:
  --config          PhiConfig(model_type='imp', vocab_size=51200, n_positions=3072, n_embd=2560, n_layer=32, n_inner=10240, n_head=32, rotary_dim=32, position_embedding_base=10000, layer_norm_epsilon=1e-05, context_window_size=3072, prefill_chunk_size=3072, n_head_kv=32, head_dim=80, tensor_parallel_shards=1, image_token_index=50296, image_token='<image>', dtype='float16', kwargs={})
  --quantization    GroupQuantize(name='q4f16_1', kind='group-quant', group_size=32, quantize_dty

#### Step 11: Push the model to Hugging Face

In [ ]:
from huggingface_hub import whoami
from pathlib import Path

# Output directory.
output_dir = "/content/mlc-imp/dist/libs"
repo_name = f"{MODEL_NAME}-{QUANTIZATION}-MLC-android"
username = whoami(token=Path("/root/.cache/huggingface/"))["name"]
repo_id = f"{username}/{repo_name}"

In [ ]:
from huggingface_hub import upload_folder, create_repo

repo_id = create_repo(repo_id, exist_ok=True).repo_id
print(output_dir)

upload_folder(
    repo_id=repo_id,
    folder_path=output_dir,
    commit_message=f"{MODEL_NAME}-{QUANTIZATION}-MLC.",
    ignore_patterns=["step_*", "epoch_*"],
)

/content/mlc-imp/dist/libs


params_shard_0.bin:   0%|          | 0.00/65.5M [00:00<?, ?B/s]

params_shard_1.bin:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

imp-v1.5-3B-196-q4f16_1-android.tar:   0%|          | 0.00/395k [00:00<?, ?B/s]

Upload 59 LFS files:   0%|          | 0/59 [00:00<?, ?it/s]

...

params_shard_6.bin:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

params_shard_7.bin:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

params_shard_8.bin:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

params_shard_9.bin:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/NSTiwari/imp-v1.5-3B-196-q4f16_1-MLC-android/commit/0cafd01fec2bde3107374fa5cf4f32da083de258', commit_message='imp-v1.5-3B-196-q4f16_1-MLC.', commit_description='', oid='0cafd01fec2bde3107374fa5cf4f32da083de258', pr_url=None, repo_url=RepoUrl('https://huggingface.co/NSTiwari/imp-v1.5-3B-196-q4f16_1-MLC-android', endpoint='https://huggingface.co', repo_type='model', repo_id='NSTiwari/imp-v1.5-3B-196-q4f16_1-MLC-android'), pr_revision=None, pr_num=None)

#### Step 12: Install Android Studio

In [ ]:
%cd /content/
!curl -O https://dl.google.com/android/repository/commandlinetools-linux-9123335_latest.zip
!sudo apt install unzip
!unzip commandlinetools-linux-9123335_latest.zip
!rm commandlinetools-linux-9123335_latest.zip
!mkdir -p android_sdk/cmdline-tools
!mv cmdline-tools android_sdk/cmdline-tools/latest

/content
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  115M  100  115M    0     0  29.7M      0  0:00:03  0:00:03 --:--:-- 29.7M
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unzip is already the newest version (6.0-26ubuntu3.2).
0 upgraded, 0 newly installed, 0 to remove and 29 not upgraded.
Archive:  commandlinetools-linux-9123335_latest.zip
 extracting: cmdline-tools/bin/avdmanager  
 extracting: cmdline-tools/bin/sdkmanager  
 extracting: cmdline-tools/bin/retrace  
 extracting: cmdline-tools/bin/apkanalyzer  
 extracting: cmdline-tools/bin/lint  
 extracting: cmdline-tools/bin/screenshot2  
 extracting: cmdline-tools/bin/profgen  
 extracting: cmdline-tools/lib/sdklib/libavdmanager_lib.jar  
 extracting: cmdline-tools/lib/sdklib/sdklib.core.jar  
 extracting: cmdline-tools/lib/common/tools.common.jar  
 extracting: 

#### Install JDK

In [ ]:
!sudo apt update
!sudo apt install -y openjdk-17-jdk

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,381 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,686 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-s

#### Set environmental variables

In [ ]:
import os
os.environ['ANDROID_HOME'] = '/content/android_sdk'
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['sdkmanager'] = '/content/android_sdk/cmdline-tools/latest/bin/sdkmanager'
os.environ['TVM_HOME'] = '/content/mlc-imp/3rdparty/tvm'
os.environ['ANDROID_NDK'] = os.path.join(os.environ['ANDROID_HOME'], 'ndk', '26.1.10909125')
os.environ['TVM_NDK_CC'] = os.path.join(os.environ['ANDROID_NDK'], 'toolchains', 'llvm', 'prebuilt', 'linux-x86_64', 'bin', 'aarch64-linux-android24-clang')

In [ ]:
!chmod +x $sdkmanager

#### Install NDK - Enter 'y' when prompted

In [ ]:
!$sdkmanager "ndk;26.1.10909125"

License android-sdk-license:
---------------------------------------
Terms and Conditions

This is the Android Software Development Kit License Agreement

1. Introduction

1.1 The Android Software Development Kit (referred to in the License Agreement as the "SDK" and specifically including the Android system files, packaged APIs, and Google APIs add-ons) is licensed to you subject to the terms of the License Agreement. The License Agreement forms a legally binding contract between you and Google in relation to your use of the SDK.

1.2 "Android" means the Android software stack for devices, as made available under the Android Open Source Project, which is located at the following URL: http://source.android.com/, as updated from time to time.

1.3 A "compatible implementation" means any Android device that (i) complies with the Android Compatibility Definition document, which can be found at the Android compatibility website (http://source.android.com/compatibility) and which may be upd

#### Compile the Android file

In [ ]:
os.rename('/content/mlc-imp/dist/libs/imp-v1.5-3B-196-q4f16_1-android.tar', '/content/mlc-imp/dist/libs/imp-v1-3b_196-q4f16_1-android.tar')

In [ ]:
%cd /content/mlc-imp/android/library
!bash prepare_libs.sh

/content/mlc-imp/android/library
+ rustup target add aarch64-linux-android
info: downloading component 'rust-std' for 'aarch64-linux-android'
info: installing component 'rust-std' for 'aarch64-linux-android'
 26.0 MiB /  26.0 MiB (100 %)  10.7 MiB/s in  2s
+ mkdir -p build/model_lib
+ python3 prepare_model_lib.py
Creating lib from ['/content/mlc-imp/dist/libs/imp-v1-3b_196-q4f16_1-android.tar']..
+ cd build
+ touch config.cmake
+ echo 'set(TVM_HOME /content/mlc-imp/3rdparty/tvm)'
+ cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_TOOLCHAIN_FILE=/content/android_sdk/ndk/26.1.10909125/build/cmake/android.toolchain.cmake -DCMAKE_INSTALL_PREFIX=. -DCMAKE_CXX_FLAGS=-O3 -DANDROID_ABI=arm64-v8a -DANDROID_NATIVE_API_LEVEL=android-24 -DANDROID_PLATFORM=android-24 -DCMAKE_FIND_ROOT_PATH_MODE_PACKAGE=ON -DANDROID_STL=c++_static -DUSE_HEXAGON_SDK=OFF -DMLC_LLM_INSTALL_STATIC_LIB=ON -DCMAKE_SKIP_INSTALL_ALL_DEPENDENCY=ON -DUSE_OPENCL=ON -DUSE_CUSTOM_LOGGING=ON
CMake Deprecation Warning at /content/andro

#### Step 15: Copy the build files

In [ ]:
!cp -r /content/mlc-imp/android/library/build /content/Mobile-VisionLM/Android_App/library/

In [ ]:
import json

# Path to the JSON file
file_path = '/content/Mobile-VisionLM/Android_App/library/src/main/assets/app-config.json'

# Read the existing JSON file
with open(file_path, 'r') as f:
    data = json.load(f)

# Modify the model_url value
data['model_list'][0]['model_url'] = 'https://huggingface.co/NSTiwari/imp-v1.5-3B-196-q4f16_1-MLC-android'

# Write the modified content back to the same JSON file
with open(file_path, 'w') as f:
    json.dump(data, f, indent=2)

print("model_url has been updated successfully.")

model_url has been updated successfully.


#### Download the Android app

In [ ]:
%cd /content/
!zip -r mobile-vision-lm-android.zip /content/Mobile-VisionLM

/content
  adding: content/Mobile-VisionLM/ (stored 0%)
  adding: content/Mobile-VisionLM/LICENSE (deflated 41%)
  adding: content/Mobile-VisionLM/Android_App/ (stored 0%)
  adding: content/Mobile-VisionLM/Android_App/gradlew (deflated 60%)
  adding: content/Mobile-VisionLM/Android_App/gradle.properties (deflated 48%)
  adding: content/Mobile-VisionLM/Android_App/gradle/ (stored 0%)
  adding: content/Mobile-VisionLM/Android_App/gradle/wrapper/ (stored 0%)
  adding: content/Mobile-VisionLM/Android_App/gradle/wrapper/gradle-wrapper.properties (deflated 35%)
  adding: content/Mobile-VisionLM/Android_App/gradle/wrapper/gradle-wrapper.jar (deflated 10%)
  adding: content/Mobile-VisionLM/Android_App/gradlew.bat (deflated 56%)
  adding: content/Mobile-VisionLM/Android_App/settings.gradle (deflated 47%)
  adding: content/Mobile-VisionLM/Android_App/README.md (deflated 2%)
  adding: content/Mobile-VisionLM/Android_App/.gitignore (deflated 43%)
  adding: content/Mobile-VisionLM/Android_App/build

In [ ]:
# Download the Android app.
from google.colab import files
files.download('/content/mobile-vision-lm-android.zip')